# NOAA GoMOFS — virtual Icechunk

Builds one [Icechunk](https://icechunk.io) store from the **NOAA NOS Gulf of Maine Operational
Forecast System (GoMOFS)** NetCDF files published in the public `noaa-nos-ofs-pds` bucket
(AWS `us-east-1`), using [VirtualiZarr](https://virtualizarr.readthedocs.io).

GoMOFS is a ROMS model. Every cycle (`t00z`, `t06z`, `t12z`, `t18z`) NOAA writes one
NetCDF per output time — 3-hourly — holding all fields on the native curvilinear
`(30 s_rho, 777 eta, 1173 xi)` grid. This notebook merges the **last `DAYS` days** of those
files into one dataset along `ocean_time`:

- the **nowcast** files `n003` and `n006` of every cycle, which tile a continuous 3-hourly
  "truth" series, and
- the **forecast** files `f003`–`f072` of the **newest cycle**, which extend it ~3 days ahead.

The store holds Zarr metadata and byte-range references only; no array data is copied. It is
a **snapshot** — this notebook has no update step, so the store does not roll forward on its own.

In [ ]:
# Dependencies. In the git repo: pip install -r requirements.txt
# Standalone (downloaded on its own from Source Cooperative), uncomment:
# !pip install -q "icechunk>=2.3" "virtualizarr>=2.8" xarray zarr obspec-utils obstore \
#     h5netcdf h5py s3fs fsspec "dask[array]" numpy boto3 matplotlib
#
# icechunk 2.x requires Python >= 3.12. The sources are NetCDF-4/HDF5, so kerchunk and
# scipy (the NetCDF-3 path) are NOT needed.

In [ ]:
import sys
# icechunk_utils.py lives at the repo root (one level up) in the git repo, or alongside this
# notebook when downloaded from Source Cooperative.
sys.path.insert(0, '..')

import datetime as dt
import re
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import fsspec
import h5py
import icechunk
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from obstore.store import S3Store
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser

warnings.filterwarnings('ignore')
print('icechunk', icechunk.__version__, '| xarray', xr.__version__)

## 1. Configuration

`RUN_WRITE` and `RUN_MIRROR` are both `False` in the committed copy, so re-running this
notebook builds the dataset in memory and validates the published store, but changes
nothing on Source Cooperative.

In [ ]:
# ------------------------------------------------------------------ source
SRC_BUCKET, SRC_REGION = 'noaa-nos-ofs-pds', 'us-east-1'
SRC = f's3://{SRC_BUCKET}'
DAYS = 5                       # how many UTC days back from today to virtualize
CONCAT_DIM = 'ocean_time'

# ------------------------------------------------------------- destination
SC_BUCKET = 'ocean-icechunks'
PROD_PREFIX = 'noaa-gomofs'         # the published store (product created in us-east-1)
CHECK_URL = 'https://data.source.coop/ocean-icechunks/noaa-gomofs'

RUN_WRITE = False
RUN_MIRROR = False

## 2. Choose the source files

Files are named `gomofs.t{HH}z.{YYYYMMDD}.fields.{n|f}{NNN}.nc` under
`gomofs/netcdf/YYYY/MM/DD/`. For the last `DAYS` days this keeps every `n*` (nowcast) file, and
only the forecast files of the **latest** cycle that has any.

The nowcast `n003`/`n006` of a cycle are valid 3 h and 0 h before its cycle time, so four
cycles a day give an unbroken 3-hourly series; the latest cycle's `f003` starts 3 h after its
cycle time, which is where the nowcast ends.

In [ ]:
fs = fsspec.filesystem('s3', anon=True)
today = dt.datetime.now(dt.timezone.utc).date()

keys = []
for d in range(DAYS):
    day = today - dt.timedelta(days=d)
    keys += fs.glob(f'{SRC_BUCKET}/gomofs/netcdf/{day:%Y/%m/%d}/*.fields.*.nc')

pat = re.compile(r'gomofs\.t(\d\d)z\.(\d{8})\.fields\.([nf])(\d{3})\.nc')
parsed = [(pat.search(k).groups(), k) for k in sorted(keys)]
nowcast = [k for (_, _, kind, _), k in parsed if kind == 'n']
forecast = [((d, h), k) for (h, d, kind, _), k in parsed if kind == 'f']
latest_cycle = max(c for c, _ in forecast)
selected = nowcast + [k for c, k in forecast if c == latest_cycle]

print(f'{len(nowcast)} nowcast + {len(selected) - len(nowcast)} forecast files '
      f'(latest cycle {latest_cycle[0]} t{latest_cycle[1]}z)')

## 3. Open the files virtually and concatenate

One `open_virtual_dataset` per file, in a thread pool. Only `ocean_time` goes in
`loadable_variables`, so it is read into memory and decoded; everything else becomes a
`ManifestArray` — a reference to a byte range in the source file.

The concat is the loose form virtual datasets need: `data_vars='minimal'` and
`coords='minimal'` so the ~85 static variables (grid, masks, scalars) are taken from the first
file instead of being broadcast along `ocean_time`, and `compat='override'` so no values are
compared.

In [ ]:
registry = ObjectStoreRegistry({SRC: S3Store(bucket=SRC_BUCKET, region=SRC_REGION, skip_signature=True)})
parser = HDFParser()

def open_one(key):
    return open_virtual_dataset(f's3://{key}', registry=registry, parser=parser,
                                loadable_variables=[CONCAT_DIM], decode_times=True)

t0 = time.perf_counter()
with ThreadPoolExecutor(16) as ex:
    parts = list(ex.map(open_one, selected))
# nowcast files only: they are the same in any build, whereas a later build's forecast cycle
# differs from the published store's, so forecast times cannot be compared across builds
nowcast_key_by_time = {p[CONCAT_DIM].values[0]: k for p, k in zip(parts, selected) if '.fields.n' in k}
parts.sort(key=lambda d: d[CONCAT_DIM].values[0])

ds = xr.concat(parts, dim=CONCAT_DIM, data_vars='minimal', coords='minimal',
               compat='override', combine_attrs='override')
print(f'{len(parts)} files virtualized in {time.perf_counter() - t0:.0f} s')
print(ds[CONCAT_DIM].values[[0, -1]], '|', ds.sizes[CONCAT_DIM], 'steps')
ds[['temp']]

## 4. Write the store

Two things are needed for a virtual store and easy to forget:

- The `VirtualChunkContainer` `url_prefix` **must end with `/`**. Without the slash no
  virtual chunk matches, silently.
- The container is saved into the repository's config, which is what lets an anonymous reader —
  who opens with no config of their own — resolve the references.

Writing is guarded by `RUN_WRITE`. The write refuses a repository that already has data:
this notebook builds a snapshot and has no append step, so rebuilding means starting from an
empty repository.

In [ ]:
def build_config():
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(
            url_prefix=f'{SRC}/',                      # trailing '/' is required
            store=icechunk.s3_store(region=SRC_REGION, anonymous=True),
        )
    )
    return config

AUTH = icechunk.containers_credentials({f'{SRC}/': icechunk.s3_anonymous_credentials()})

if RUN_WRITE:
    from icechunk_utils import open_source_icechunk_repo

    repo, storage, _creds, time_left = open_source_icechunk_repo(
        bucket=SC_BUCKET, prefix=PROD_PREFIX, config=build_config(), min_minutes_left=15)
    if repo is None:
        raise SystemExit('credentials too close to expiry; see the message above')
    repo = icechunk.Repository.open(storage, config=build_config(),
                                    authorize_virtual_chunk_access=AUTH)
    if len(list(repo.ancestry(branch='main'))) > 1:
        raise SystemExit('repository already has commits; this notebook only builds a first snapshot')
    repo.save_config()
    session = repo.writable_session('main')
    ds.vz.to_icechunk(session.store)
    snap = session.commit(f'GoMOFS {DAYS}d nowcast + forecast cycle {latest_cycle[0]}t{latest_cycle[1]}z')
    print('committed', snap)
else:
    print('RUN_WRITE is False: nothing written')

## 5. Validate the published store

From the consumer's read path — an anonymous HTTPS open of the public URL, with no credentials
and nothing left over from the write session above. The arrays live in NOAA's bucket, outside
the store, so Icechunk has to be told it may fetch them; the container is in the saved config,
so a reader can discover it rather than knowing the bucket in advance.

In [ ]:
repo_ro = icechunk.Repository.open(icechunk.http_storage(CHECK_URL))
containers = list(repo_ro.config.virtual_chunk_containers or [])
print('virtual chunk containers in the saved config:', containers)

auth = icechunk.containers_credentials({p: icechunk.s3_anonymous_credentials() for p in containers})
store = repo_ro.reopen(authorize_virtual_chunk_access=auth).readonly_session('main').store
published = xr.open_zarr(store, consolidated=False, chunks=None)
print('latest commit:', next(iter(repo_ro.ancestry(branch='main'))).message)
published[['temp']]

In [ ]:
assert published.temp.dims == ('ocean_time', 's_rho', 'eta_rho', 'xi_rho')
assert dict(published.temp.sizes) | {'ocean_time': 0} == {'ocean_time': 0, 's_rho': 30, 'eta_rho': 777, 'xi_rho': 1173}
assert published.indexes['ocean_time'].is_monotonic_increasing
steps = np.unique(np.diff(published.ocean_time.values).astype('timedelta64[h]').astype(int))
assert list(steps) == [3], steps                    # continuous 3-hourly series, no gaps
assert {'lon_rho', 'lat_rho'} <= set(published.coords)
print(published.sizes['ocean_time'], 'steps,', published.ocean_time.values[0], '->', published.ocean_time.values[-1])

## 6. Compare against the source files

The store holds no data, so "is it correct" means "does a read through the reference return the
bytes in the file". This reads the surface temperature (`s_rho` = last index) at the first,
middle and last **nowcast** time (at least 12 h before the store was committed) that the published store shares with the in-memory build above, from both the
store and the source NetCDF, and compares them element by element, NaN for NaN. The source marks land with the
 1e37, which xarray turns into NaN, so the source values are masked the same way.

In [ ]:
# A time that was still in the future when the store was committed is a forecast there but
# may be a nowcast in a later build, so only compare times safely before the commit.
committed = next(iter(repo_ro.ancestry(branch='main'))).written_at
cutoff = np.datetime64(committed.astimezone(dt.timezone.utc).replace(tzinfo=None)) - np.timedelta64(12, 'h')
common = sorted(t for t in set(published.ocean_time.values) & set(nowcast_key_by_time) if t <= cutoff)
assert common, 'published store and this build share no nowcast times; rerun closer to the build date'
picks = [common[0], common[len(common) // 2], common[-1]]

for t in picks:
    with h5py.File(fs.open(nowcast_key_by_time[t]), 'r') as h:
        fill = h['temp'].attrs['_FillValue']          # land is 1e37 in the file; xarray masks it to NaN
        src = h['temp'][0, -1]
    src = np.where(src == fill, np.nan, src)
    got = published.temp.sel(ocean_time=t).isel(s_rho=-1).values
    assert np.array_equal(src, got, equal_nan=True), t
    print(f'  {np.datetime_as_string(t, unit="h")}: surface temp identical ({src.size} cells)')

## 7. A look at the data

Surface temperature at the most recent nowcast time, read through the virtual references. The
curvilinear `lon_rho`/`lat_rho` are used directly; land is `NaN`.

In [ ]:
latest_truth = max(t for t in published.ocean_time.values
                   if t <= np.datetime64(dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)))
sst = published.temp.sel(ocean_time=latest_truth).isel(s_rho=-1)
fig, ax = plt.subplots(figsize=(8, 5.5), constrained_layout=True)
m = ax.pcolormesh(published.lon_rho, published.lat_rho, sst, cmap='turbo', shading='auto')
fig.colorbar(m, ax=ax, label='°C')
ax.set_title(f'GoMOFS surface temperature, {np.datetime_as_string(latest_truth, unit="h")}Z')
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
print(f'range {float(np.nanmin(sst)):.1f} to {float(np.nanmax(sst)):.1f} °C')

## 8. Mirror the documentation to the store

Source Cooperative serves the product's prefix as plain files, so the README, the dependency
floors, the credential helper and this notebook are uploaded beside the store.

Mirror **only from merged `main`** — uploading from a branch is how the published copies drift
out of step with the repository.

In [ ]:
if RUN_MIRROR:
    from pathlib import Path
    import boto3
    from icechunk_utils import get_source_credentials

    creds, _exp = get_source_credentials()
    s3 = boto3.client(
        's3',
        endpoint_url=creds['endpoint_url'],
        aws_access_key_id=creds['aws_access_key_id'],
        aws_secret_access_key=creds['aws_secret_access_key'],
        aws_session_token=creds['aws_session_token'],
        region_name=creds['region_name'],
    )
    DOCS = ['README.md', 'requirements.txt', 'noaa-gomofs-icechunk-sc.ipynb', '../icechunk_utils.py']
    for rel in DOCS:
        path = Path(rel)
        ctype = {'.md': 'text/markdown', '.txt': 'text/plain', '.py': 'text/x-python',
                 '.ipynb': 'application/x-ipynb+json'}[path.suffix]
        s3.put_object(Bucket=SC_BUCKET, Key=f'{PROD_PREFIX}/{path.name}',
                      Body=path.read_bytes(), ContentType=ctype)
        print('uploaded', path.name)
else:
    print('RUN_MIRROR is False: nothing uploaded')